# CodeMoldova — Week 2 Thursday — Answer key

Gemini Flash chat agent lab. Try the starter notebook first.


## 2 · Key check


In [ ]:
import os
api_key = os.environ.get("GEMINI_API_KEY")
assert api_key, "Set GEMINI_API_KEY in the terminal first"
print("Key OK:", api_key[:4] + "...")


## 3–4 · Import & client


In [ ]:
from google import genai
client = genai.Client()
print("Client ready")


## 5 · One-shot


In [ ]:
response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents="In one short paragraph, what is an API?",
)
print(response.text)


## 6–7 · Chat loop with system prompt


In [ ]:
SYSTEM = "You are a patient Python tutor for beginners. Keep answers under 80 words."

chat = client.chats.create(
    model="gemini-2.0-flash",
    config={"system_instruction": SYSTEM, "max_output_tokens": 300},
)

print("Chat ready — type quit to exit\n")
while True:
    user_text = input("You: ").strip()
    if user_text.lower() in ("quit", "exit", "q"):
        break
    if not user_text:
        print("(empty — type something)")
        continue
    reply = chat.send_message(user_text)
    print(f"\nBot: {reply.text}\n")


## 8 · Save chat log while chatting


In [ ]:
from pathlib import Path
import json
from google import genai

Path("data").mkdir(exist_ok=True)
client = genai.Client()
SYSTEM = "You are a movie critic. Only discuss films. Under 60 words."
chat = client.chats.create(
    model="gemini-2.0-flash",
    config={"system_instruction": SYSTEM, "max_output_tokens": 200},
)

chat_log = []
print("Log chat — quit to save\n")
while True:
    user_text = input("You: ").strip()
    if user_text.lower() in ("quit", "exit", "q"):
        break
    if not user_text:
        continue
    chat_log.append({"role": "user", "content": user_text})
    reply = chat.send_message(user_text)
    chat_log.append({"role": "assistant", "content": reply.text})
    print(f"\nBot: {reply.text}\n")

path = Path("data/chat_log.json")
path.write_text(json.dumps(chat_log, indent=2), encoding="utf-8")
print(f"Saved {len(chat_log)} messages to {path}")


## 8b · Read transcript from disk


In [ ]:
from pathlib import Path
import json

log = json.loads(Path("data/chat_log.json").read_text(encoding="utf-8"))
for i, msg in enumerate(log, 1):
    who = "You" if msg["role"] == "user" else "Bot"
    print(f"{i}. {who}: {msg['content'][:200]}{'...' if len(msg['content']) > 200 else ''}")
